# Predicting House Sale Prices with Automation-Driven ML

**Author:** Alberto Diaz Durana  
**Date:** March 2026  
**Course:** AI in Data Science  

## Objective

This notebook explores how automation tools change the machine learning workflow. Using the Ames Housing dataset (~2,900 homes, 82 features), we build a regression model to predict sale prices — but the real focus is on productivity. How much does automated EDA, feature engineering, and model selection actually improve over a manual baseline?

We compare a hand-built linear regression against PyCaret's automated pipeline to quantify what automation delivers and where human judgment remains essential.

## Dataset

The Ames Housing dataset contains detailed information about residential properties in Ames, Iowa, including lot characteristics, building quality, room counts, garage details, and sale conditions. The target variable is `SalePrice`.

## Notebook Structure

1. **Setup & Data Loading** — Environment, imports, initial inspection
2. **Automated EDA** — YData Profiling and SweetViz reports, findings synthesis
3. **Data Preprocessing** — Missing values, encoding, outliers
4. **Automated Feature Engineering** — Autofeat (primary) and Featuretools (contrast)
5. **Modeling** — Manual baseline vs. PyCaret automated pipeline with tuning
6. **Interpretation** — SHAP-based feature importance
7. **Reflections** — What automation improved and where human judgment mattered

In [3]:
# Cell 2 - Installs + Imports

import subprocess
import sys

# Install dependencies (for Colab or fresh environments)
packages = [
    'ydata-profiling', 'sweetviz', 'autofeat', 'featuretools',
    'pycaret', 'shap'
]
for pkg in packages:
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '-q', pkg],
        stdout=subprocess.DEVNULL
    )

import os
import warnings
import numpy as np
import pandas as pd

# EDA
from ydata_profiling import ProfileReport
import sweetviz as sv

# Feature Engineering
from autofeat import AutoFeatRegressor
import featuretools as ft

# Modeling
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Interpretation
import shap

# Output directories
os.makedirs('outputs', exist_ok=True)

warnings.filterwarnings('ignore')

print(f'Python: {sys.version}')
print(f'NumPy: {np.__version__}')
print(f'Pandas: {pd.__version__}')

Python: 3.10.12 (main, Jan 26 2026, 14:55:28) [GCC 11.4.0]
NumPy: 1.26.4
Pandas: 2.1.4


In [4]:
# Cell 3 - Data Loading + Initial Inspection

import os

LOCAL_PATH = '../data/AmesHousing.csv'
GDRIVE_PATH = '/content/drive/MyDrive/data/AmesHousing.csv'
GDRIVE_URL = 'https://drive.google.com/uc?id=1-KhJhFnj5vmESlknOtpra-2E86_thEWO'

if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
    print(f'Loaded from local path: {LOCAL_PATH}')
elif os.path.exists(GDRIVE_PATH):
    df = pd.read_csv(GDRIVE_PATH)
    print(f'Loaded from Google Drive: {GDRIVE_PATH}')
else:
    import gdown
    gdown.download(GDRIVE_URL, 'AmesHousing.csv', quiet=False)
    df = pd.read_csv('AmesHousing.csv')
    print('Downloaded from Google Drive')

print(f'Shape: {df.shape}')
print(f'\nColumn types:\n{df.dtypes.value_counts()}')
print(f'\nTarget (SalePrice):')
print(df['SalePrice'].describe())

print(df.head(3))

Loaded from local path: ../data/AmesHousing.csv
Shape: (2930, 82)

Column types:
object     43
int64      28
float64    11
Name: count, dtype: int64

Target (SalePrice):
count      2930.000000
mean     180796.060068
std       79886.692357
min       12789.000000
25%      129500.000000
50%      160000.000000
75%      213500.000000
max      755000.000000
Name: SalePrice, dtype: float64
   Order        PID  MS SubClass MS Zoning  Lot Frontage  Lot Area Street  \
0      1  526301100           20        RL         141.0     31770   Pave   
1      2  526350040           20        RH          80.0     11622   Pave   
2      3  526351010           20        RL          81.0     14267   Pave   

  Alley Lot Shape Land Contour  ... Pool Area Pool QC  Fence Misc Feature  \
0   NaN       IR1          Lvl  ...         0     NaN    NaN          NaN   
1   NaN       Reg          Lvl  ...         0     NaN  MnPrv          NaN   
2   NaN       IR1          Lvl  ...         0     NaN    NaN         Gar2 

## Automated EDA

With 82 features, manually profiling each variable would be tedious and error-prone. Instead, we use two complementary automated EDA tools. YData Profiling generates a comprehensive statistical report — distributions, missing values, correlations, and alerts for data quality issues. SweetViz adds a target-aware perspective, highlighting how each feature relates to `SalePrice` and ranking them by association strength.

Running both gives us a more complete picture than either tool alone: YData catches structural issues (high cardinality, constant columns, duplicates) while SweetViz focuses on predictive relevance.

In [13]:
# Cell 5 - YData Profiling Report

profile = ProfileReport(
    df,
    title='Ames Housing - YData Profiling',
    minimal=True,
    progress_bar=True
)



In [14]:
profile.to_file('outputs/ydata_report.html')
print('Report saved to outputs/ydata_report.html')
#profile.to_notebook_iframe()

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 82/82 [00:00<00:00, 1644.19it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

### YData Profiling - Key Takeaways

The automated report flagged 40 alerts across the dataset. Most fall into three categories that will shape our preprocessing strategy.

**Missing values are mostly structural, not errors.** Features like Pool QC (99.6% missing), Misc Feature (96.4%), Alley (93.2%), and Fence (80.5%) use `NaN` to indicate the property lacks that feature; a pool, a special feature, an alley, a fence. These are informative absences, not data quality problems. The garage group (~5.4% missing) and basement group (~2.7%) follow the same pattern: no garage or no basement. Lot Frontage (16.7%) is the main genuinely missing value that needs imputation.

**The strongest predictors are quality and size.** Overall Qual dominates with a 0.80 correlation to SalePrice, followed by Gr Liv Area (0.71), Garage Cars (0.65), Garage Area (0.64), and Total Bsmt SF (0.63). Year Built (0.56) also matters; newer homes sell for more. These top features will guide our feature selection for Autofeat later.

**ID columns must go.** Order and PID are unique identifiers with no predictive value; we'll drop them before modeling.


In [ ]:
# Cell 6 - SweetViz Report

sv_config = sv.FeatureConfig(skip=['Order', 'PID'])

report = sv.analyze(
    df,
    target_feat='SalePrice',
    feat_cfg=sv_config,
    pairwise_analysis='off'
)

report.show_html('outputs/sweetviz_report.html', open_browser=False)
print('Report saved to outputs/sweetviz_report.html')


                                             |          | [  0%]   00:00 -> (? left)

Report outputs/sweetviz_report.html was generated.
Report saved to outputs/sweetviz_report.html


### EDA Findings Synthesis

Both tools confirmed the same core picture, but from different angles. YData Profiling flagged 40 alerts, primarily missing values and zero-heavy columns, giving us a structured quality checklist. SweetViz added a target-aware lens, visually ranking each feature's relationship with SalePrice and showing how distributions shift across price ranges.

**What each tool caught that the other emphasized less.** YData excels at surfacing data quality issues systematically: it flagged that Pool QC (99.6%), Misc Feature (96.4%), and Alley (93.2%) are almost entirely missing, and that Misc Val has extreme skewness (γ₁ = 22.0). SweetViz made it easier to visually compare how categorical features like Neighborhood, Kitchen Qual, and Exter Qual relate to SalePrice, something that's harder to see in a correlation matrix limited to numeric features. SweetViz's detailed stats also revealed Garage Yr Blt has a max of 2,207, a clear data entry error that needs correction, and confirmed extreme outliers in Lot Area (skewness 12.8, kurtosis 265).

**Preprocessing decisions informed by these reports:**
- **Drop:** Order, PID (IDs); Pool QC, Misc Feature, Alley, Fence (>80% missing, low signal); Low Qual Fin SF, 3Ssn Porch (>98% zeros)
- **Fix:** Garage Yr Blt max=2,207 (data entry error, cap or impute)
- **Fill with "None"/0:** Garage group (~5.4%), Basement group (~2.7%), Fireplace Qu (48.5%), Mas Vnr Type/Area, structural absences, not data errors
- **Impute:** Lot Frontage (16.7% genuinely missing), median by Neighborhood
- **Watch:** Lot Area outliers (max 215k vs. median 9k), SalePrice right skew (may need log transform)
- **Feature selection candidates:** Overall Qual (0.80), Gr Liv Area (0.71), Garage Cars (0.65), Total Bsmt SF (0.63), 1st Flr SF (0.62), Year Built (0.56)


### Focused EDA: Surfacing What Needs Action

The automated reports (YData Profiling, SweetViz) generated comprehensive profiles for all 82 features and are saved as HTML files for reference. Rather than trying to analyze every plot and table, the next few cells compute targeted summaries that surface only the extremes and actionable findings: missing values grouped by handling strategy, top correlations with anomaly flags, and near-zero-variance columns. This keeps the analysis focused on decisions that affect preprocessing, not on describing every feature.


In [ ]:
# Cell 7 - Missing Values Analysis

missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)

print("=== MISSING VALUES BY STRATEGY ===\n")

print("DROP (>80% missing, low signal):")
drop = missing_pct[missing_pct > 80]
for col, pct in drop.items():
    print(f"  {col:20s} {pct}%")

print("\nFILL WITH 'None'/0 (structural absence):")
fill_cols = ['Fireplace Qu', 'Garage Type', 'Garage Yr Blt', 'Garage Finish',
             'Garage Qual', 'Garage Cond', 'Bsmt Qual', 'Bsmt Cond',
             'Bsmt Exposure', 'BsmtFin Type 1', 'BsmtFin Type 2', 'Mas Vnr Type']
for col in fill_cols:
    if col in missing_pct:
        print(f"  {col:20s} {missing_pct[col]}%")

print("\nIMPUTE (genuinely missing):")
impute_cols = ['Lot Frontage']
for col in impute_cols:
    if col in missing_pct:
        print(f"  {col:20s} {missing_pct[col]}%")

remaining = missing_pct.drop(
    [c for c in drop.index.tolist() + fill_cols + impute_cols if c in missing_pct.index],
    errors='ignore'
)
if len(remaining) > 0:
    print("\nOTHER (<1% missing, safe to drop rows or fill):")
    for col, pct in remaining.items():
        print(f"  {col:20s} {pct}%")


=== MISSING VALUES BY STRATEGY ===

DROP (>80% missing, low signal):
  Pool QC              99.6%
  Misc Feature         96.4%
  Alley                93.2%
  Fence                80.5%

FILL WITH 'None'/0 (structural absence):
  Fireplace Qu         48.5%
  Garage Type          5.4%
  Garage Yr Blt        5.4%
  Garage Finish        5.4%
  Garage Qual          5.4%
  Garage Cond          5.4%
  Bsmt Qual            2.7%
  Bsmt Cond            2.7%
  Bsmt Exposure        2.8%
  BsmtFin Type 1       2.7%
  BsmtFin Type 2       2.8%
  Mas Vnr Type         60.6%

IMPUTE (genuinely missing):
  Lot Frontage         16.7%

OTHER (<1% missing, safe to drop rows or fill):
  Mas Vnr Area         0.8%
  Bsmt Half Bath       0.1%
  Bsmt Full Bath       0.1%
  BsmtFin SF 1         0.0%
  Garage Cars          0.0%
  Garage Area          0.0%
  Total Bsmt SF        0.0%
  Bsmt Unf SF          0.0%
  BsmtFin SF 2         0.0%
  Electrical           0.0%


In [ ]:
# Cell 8 - Top Correlations + Anomaly Flags

numeric = df.select_dtypes(include='number')
corr = numeric.corr()['SalePrice'].drop('SalePrice').abs().sort_values(ascending=False)

print("=== TOP 15 CORRELATIONS WITH SalePrice ===\n")
print(f"{'Feature':25s} {'Corr':>6s}  {'Min':>10s} {'Max':>10s} {'Skew':>6s}  Flags")
print("-" * 85)

for col, c in corr.head(15).items():
    col_data = df[col]
    skew = col_data.skew()
    flags = []
    if col_data.isnull().sum() > 0:
        flags.append(f"missing:{col_data.isnull().sum()}")
    if abs(skew) > 3:
        flags.append(f"high skew")
    if col_data.max() > col_data.quantile(0.99) * 3:
        flags.append("extreme max")
    flag_str = ", ".join(flags) if flags else ""
    print(f"  {col:23s} {c:6.3f}  {col_data.min():10.0f} {col_data.max():10.0f} {skew:6.2f}  {flag_str}")

print("\n=== KNOWN ANOMALY ===")
print(f"  Garage Yr Blt max = {df['Garage Yr Blt'].max():.0f} (data entry error)")


=== TOP 15 CORRELATIONS WITH SalePrice ===

Feature                     Corr         Min        Max   Skew  Flags
-------------------------------------------------------------------------------------
  Overall Qual             0.799           1         10   0.19  
  Gr Liv Area              0.707         334       5642   1.27  
  Garage Cars              0.648           0          5  -0.22  missing:1
  Garage Area              0.640           0       1488   0.24  missing:1
  Total Bsmt SF            0.632           0       6110   1.16  missing:1
  1st Flr SF               0.622         334       5095   1.47  
  Year Built               0.558        1872       2010  -0.60  
  Full Bath                0.546           0          4   0.17  
  Year Remod/Add           0.533        1950       2010  -0.45  
  Garage Yr Blt            0.527        1895       2207  -0.38  missing:159
  Mas Vnr Area             0.508           0       1600   2.61  missing:23
  TotRms AbvGrd            0.495     

In [ ]:
# Cell 9 - Near-Zero-Variance Check

print("=== NEAR-ZERO-VARIANCE FEATURES (>95% one value) ===\n")
print(f"{'Feature':25s} {'Dominant %':>10s}  {'Dominant Value'}")
print("-" * 60)

nzv = []
for col in df.columns:
    top_pct = df[col].value_counts(normalize=True, dropna=False).iloc[0] * 100
    if top_pct > 95:
        top_val = df[col].value_counts(dropna=False).index[0]
        nzv.append((col, top_pct, top_val))

nzv.sort(key=lambda x: x[1], reverse=True)
for col, pct, val in nzv:
    print(f"  {col:23s} {pct:9.1f}%  {val}")

print(f"\n{len(nzv)} features with >95% single value, candidates for dropping.")


=== NEAR-ZERO-VARIANCE FEATURES (>95% one value) ===

Feature                   Dominant %  Dominant Value
------------------------------------------------------------
  Utilities                    99.9%  AllPub
  Street                       99.6%  Pave
  Pool Area                    99.6%  0
  Pool QC                      99.6%  nan
  Condition 2                  99.0%  Norm
  3Ssn Porch                   98.7%  0
  Low Qual Fin SF              98.6%  0
  Roof Matl                    98.5%  CompShg
  Heating                      98.5%  GasA
  Misc Val                     96.5%  0
  Misc Feature                 96.4%  nan
  Kitchen AbvGr                95.4%  1
  Land Slope                   95.2%  Gtl

13 features with >95% single value, candidates for dropping.


### EDA Summary: Preprocessing Decisions

The three focused analyses above give us a clear preprocessing plan.

**Features to drop** (high missing, near-zero-variance, or no predictive value): Order, PID, Pool QC, Pool Area, Misc Feature, Misc Val, Alley, Fence, Utilities, Street, Condition 2, 3Ssn Porch, Low Qual Fin SF, Roof Matl, Heating. These are either almost entirely one value or overwhelmingly missing.

**Structural absences to fill** (not missing data, just "feature not present"): Fireplace Qu, Garage group (Type, Yr Blt, Finish, Qual, Cond), Basement group (Qual, Cond, Exposure, BsmtFin Type 1/2), Mas Vnr Type/Area. Fill categorical columns with "None", numeric with 0.

**Genuinely missing to impute:** Lot Frontage (16.7%), median by Neighborhood. The remaining <1% missing values (Electrical, Bsmt/Garage numerics) can be filled with mode or median.

**Anomalies to fix:** Garage Yr Blt max=2,207 (data entry error), BsmtFin SF 1 max=5,644 (potential outlier, verify in HTML report).

**Top predictors for feature engineering:** Overall Qual (0.80), Gr Liv Area (0.71), Garage Cars (0.65), Garage Area (0.64), Total Bsmt SF (0.63), 1st Flr SF (0.62).


----------------

### A Note on Automated EDA: More Output Is Not More Insight

YData Profiling and SweetViz each generated full HTML reports covering all 82 features with distributions, statistics, alerts, and associations. These reports are saved and available for reference, but every finding in our focused analysis cells came directly from the raw dataframe, not from parsing these reports.

This is worth acknowledging honestly. We ran two automated tools, generated hundreds of plots and statistics, then wrote three short cells that extracted the same information more usefully by organizing it around decisions (what to drop, what to fill, what to impute). The automated reports were an intermediate step we had to re-process anyway.

The tools are not useless, they work well as a safety net to check whether we missed something, and as browsable reference files when we want to validate a specific feature in detail. But treating their output as the analysis itself is a trap. With 82 features, scrolling through comprehensive reports is closer to skimming than analyzing, and important patterns get buried alongside irrelevant ones.

The real productivity gain came from writing targeted code that surfaces only the extremes and groups findings by action. That took a few minutes and produced three concise, actionable outputs. The lesson: automation tools are reference material, not a substitute for knowing what questions to ask the data.


-------------

## Data Preprocessing

The EDA analysis identified four categories of action. First, 15 columns get dropped entirely: the two ID columns (Order, PID), four with over 80% missing values (Pool QC, Misc Feature, Alley, Fence), and nine near-zero-variance features where almost every row has the same value (Utilities, Street, Pool Area, Condition 2, 3Ssn Porch, Low Qual Fin SF, Roof Matl, Heating, Misc Val).

Second, structural absences get filled rather than treated as missing data. Categorical features like Fireplace Qu, the Garage group, and the Basement group use NaN to mean "this property doesn't have that feature," so we fill them with "None." Their corresponding numeric columns (Garage Area, Total Bsmt SF, etc.) get filled with 0.

Third, Lot Frontage (16.7% missing) is genuinely missing and gets imputed using the median within each Neighborhood, since lot frontage varies significantly by location. The Garage Yr Blt anomaly (max=2,207) gets set to 0 as well, since the only reasonable interpretation is that it's a data entry error for a property with a garage.

Finally, the handful of remaining missing values (<1%) get filled with their column's mode.


In [15]:
# Cell 10 - Data Preprocessing

print(f"Starting shape: {df.shape}")

# 1. Drop high-missing, near-zero-variance, and ID columns
drop_cols = [
    'Order', 'PID',
    'Pool QC', 'Pool Area', 'Misc Feature', 'Misc Val',
    'Alley', 'Fence', 'Utilities', 'Street', 'Condition 2',
    '3Ssn Porch', 'Low Qual Fin SF', 'Roof Matl', 'Heating'
]
df = df.drop(columns=drop_cols)
print(f"After dropping {len(drop_cols)} columns: {df.shape}")

# 2. Fix Garage Yr Blt anomaly (2207 -> NaN, will be filled below)
df.loc[df['Garage Yr Blt'] > 2025, 'Garage Yr Blt'] = np.nan

# 3. Fill structural absences
cat_none = ['Fireplace Qu', 'Garage Type', 'Garage Finish', 'Garage Qual',
            'Garage Cond', 'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure',
            'BsmtFin Type 1', 'BsmtFin Type 2', 'Mas Vnr Type']
for col in cat_none:
    df[col] = df[col].fillna('None')

num_zero = ['Mas Vnr Area', 'Garage Yr Blt', 'BsmtFin SF 1', 'BsmtFin SF 2',
            'Bsmt Unf SF', 'Total Bsmt SF', 'Bsmt Full Bath', 'Bsmt Half Bath',
            'Garage Cars', 'Garage Area']
for col in num_zero:
    df[col] = df[col].fillna(0)

# 4. Impute Lot Frontage by Neighborhood median
df['Lot Frontage'] = df.groupby('Neighborhood')['Lot Frontage'].transform(
    lambda x: x.fillna(x.median())
)

# 5. Fill remaining missing with mode
remaining_missing = df.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]
for col in remaining_missing.index:
    df[col] = df[col].fillna(df[col].mode()[0])

print(f"Remaining missing values: {df.isnull().sum().sum()}")
print(f"Final shape: {df.shape}")


Starting shape: (2930, 82)
After dropping 15 columns: (2930, 67)
Remaining missing values: 0
Final shape: (2930, 67)


## Automated Feature Engineering

We use Autofeat to generate interaction and transformation features from the top numeric predictors identified in EDA. Autofeat fits a regression internally and keeps only the generated features that improve the model, so it acts as both generator and selector. The key constraint is `feateng_steps=1`, since higher values cause combinatorial runtime explosion.

We first split the data to ensure Autofeat fits only on training data, preventing information leakage into the test set.


In [16]:
# Cell 11 - Train/Test Split + Autofeat Feature Engineering

from sklearn.model_selection import train_test_split

target = 'SalePrice'
X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# Select top numeric features for Autofeat
top_numeric = [
    'Overall Qual', 'Gr Liv Area', 'Garage Cars', 'Garage Area',
    'Total Bsmt SF', '1st Flr SF', 'Year Built', 'Full Bath',
    'Year Remod/Add', 'TotRms AbvGrd'
]

X_train_num = X_train[top_numeric].copy()
X_test_num = X_test[top_numeric].copy()

print(f"Autofeat input: {len(top_numeric)} features")
print("Running Autofeat (feateng_steps=1)...")

afreg = AutoFeatRegressor(feateng_steps=1, verbose=1)
X_train_af = afreg.fit_transform(X_train_num, y_train)
X_test_af = afreg.transform(X_test_num)

new_features = [c for c in X_train_af.columns if c not in top_numeric]
print(f"\nOriginal features: {len(top_numeric)}")
print(f"New features generated: {len(new_features)}")
print(f"Total after Autofeat: {X_train_af.shape[1]}")

if new_features:
    print("\nGenerated features:")
    for f in new_features:
        print(f"  {f}")


2026-03-04 19:58:13,328 INFO: [AutoFeat] The 1 step feature engineering process could generate up to 70 features.
2026-03-04 19:58:13,330 INFO: [AutoFeat] With 2344 data points this new feature matrix would use about 0.00 gb of space.
2026-03-04 19:58:13,333 INFO: [feateng] Step 1: transformation of original features


Train: (2344, 66), Test: (586, 66)
Autofeat input: 10 features
Running Autofeat (feateng_steps=1)...


2026-03-04 19:58:16,091 INFO: [feateng] Generated 42 transformed features from 10 original features - done.
2026-03-04 19:58:16,094 INFO: [feateng] Generated altogether 48 new features in 1 steps
2026-03-04 19:58:16,095 INFO: [feateng] Removing correlated features, as well as additions at the highest level
2026-03-04 19:58:16,117 INFO: [feateng] Generated a total of 20 additional features


[featsel] Scaling data...

2026-03-04 19:58:20,286 INFO: [featsel] Feature selection run 1/5


done.


2026-03-04 19:58:23,544 INFO: [featsel] Feature selection run 2/5
2026-03-04 19:58:23,975 INFO: [featsel] Feature selection run 3/5
2026-03-04 19:58:24,629 INFO: [featsel] Feature selection run 4/5
2026-03-04 19:58:25,276 INFO: [featsel] Feature selection run 5/5
2026-03-04 19:58:26,194 INFO: [featsel] 21 features after 5 feature selection runs
2026-03-04 19:58:26,214 INFO: [featsel] 20 features after correlation filtering
2026-03-04 19:58:26,464 INFO: [featsel] 19 features after noise filtering
2026-03-04 19:58:26,481 INFO: [AutoFeat] Computing 10 new features.


2026-03-04 19:58:28,009 INFO: [AutoFeat]    10/   10 new features ...done.
2026-03-04 19:58:28,012 INFO: [AutoFeat] Final dataframe with 20 feature columns (10 new).
2026-03-04 19:58:28,013 INFO: [AutoFeat] Training final regression model.


2026-03-04 19:58:28,212 INFO: [AutoFeat] Trained model: largest coefficients:
2026-03-04 19:58:28,216 INFO: -1342181.7616481765
2026-03-04 19:58:28,218 INFO: 110510.831981 * 1/OverallQual
2026-03-04 19:58:28,219 INFO: 22663.833741 * Overall Qual
2026-03-04 19:58:28,222 INFO: -15935.847697 * Full Bath
2026-03-04 19:58:28,223 INFO: -2438.860901 * TotRms AbvGrd
2026-03-04 19:58:28,224 INFO: 772.018258 * exp(FullBath)
2026-03-04 19:58:28,225 INFO: 481.835638 * GarageCars**3
2026-03-04 19:58:28,227 INFO: 347.038630 * Year Remod/Add
2026-03-04 19:58:28,228 INFO: 317.184399 * Year Built
2026-03-04 19:58:28,229 INFO: -56.942575 * Total Bsmt SF
2026-03-04 19:58:28,229 INFO: 31.274455 * Gr Liv Area
2026-03-04 19:58:28,230 INFO: 11.178151 * Garage Area
2026-03-04 19:58:28,231 INFO: 10.571906 * 1st Flr SF
2026-03-04 19:58:28,232 INFO: 0.070854 * TotalBsmtSF**2
2026-03-04 19:58:28,233 INFO: 0.000013 * GarageArea**3
2026-03-04 19:58:28,234 INFO: -0.000013 * TotalBsmtSF**3
2026-03-04 19:58:28,243 INF

[AutoFeat]     9/   10 new features
Original features: 10
New features generated: 10
Total after Autofeat: 20

Generated features:
  GarageCars**3
  GrLivArea**3
  x1stFlrSF**3
  exp(FullBath)
  1/OverallQual
  TotalBsmtSF**3
  TotalBsmtSF**2
  1/x1stFlrSF
  1/GrLivArea
  GarageArea**3


In [19]:
# Cell 12 - Featuretools Contrast (Brief)

es = ft.EntitySet(id='ames')
es = es.add_dataframe(
    dataframe_name='houses',
    dataframe=X_train.reset_index(),
    index='index'
)

# Only transform primitives work on a single table (no aggregations)
ft_features = ft.dfs(
    entityset=es,
    target_dataframe_name='houses',
    trans_primitives=['multiply_numeric', 'add_numeric'],
    max_depth=1,
    verbose=False
)[0]

print(f"Original columns: {X_train.shape[1]}")
print(f"Featuretools output: {ft_features.shape[1]}")
print(f"New features generated: {ft_features.shape[1] - X_train.shape[1]}")



Original columns: 66
Featuretools output: 1058
New features generated: 992


### Feature Engineering Assessment

Cell 11 used Autofeat on the 10 strongest numeric predictors. It generated candidate features (transforms like cubes, inverses, exponentials), then selected only the 10 that improved its internal regression model (R²=0.85). Result: 20 features total, each validated against the target.

Cell 12 used Featuretools on all 66 columns. It generated 992 features by combining every numeric pair with multiply and add operations, with no filtering or evaluation. This is what Featuretools does on a single flat table: it can only apply transforms, not the aggregation primitives it was designed for in relational datasets with linked tables.

Autofeat gave us a small, validated feature set. Featuretools gave us a massive, unfiltered one. For modeling, we proceed with Autofeat's 20 features.


## Modeling

We compare three approaches to quantify what automation adds at each stage. First, a manual sklearn LinearRegression on the plain numeric features, the simplest possible baseline. Second, the same model on Autofeat's 20 features, to isolate the impact of automated feature engineering. Third, PyCaret's automated pipeline on the full dataset including categoricals, with model comparison and tuning.


In [20]:
# Cell 13 - Manual Baseline (LinearRegression, no engineered features)

X_train_num_plain = X_train[top_numeric].copy()
X_test_num_plain = X_test[top_numeric].copy()

lr_baseline = LinearRegression()
lr_baseline.fit(X_train_num_plain, y_train)
y_pred_baseline = lr_baseline.predict(X_test_num_plain)

rmse_base = np.sqrt(mean_squared_error(y_test, y_pred_baseline))
mae_base = mean_absolute_error(y_test, y_pred_baseline)
r2_base = r2_score(y_test, y_pred_baseline)

print("=== BASELINE: LinearRegression on 10 numeric features ===\n")
print(f"  RMSE:  ${rmse_base:,.0f}")
print(f"  MAE:   ${mae_base:,.0f}")
print(f"  R²:    {r2_base:.4f}")


=== BASELINE: LinearRegression on 10 numeric features ===

  RMSE:  $39,568
  MAE:   $24,855
  R²:    0.8047


In [21]:
# Cell 14 - LinearRegression with Autofeat Features

lr_autofeat = LinearRegression()
lr_autofeat.fit(X_train_af, y_train)
y_pred_af = lr_autofeat.predict(X_test_af)

rmse_af = np.sqrt(mean_squared_error(y_test, y_pred_af))
mae_af = mean_absolute_error(y_test, y_pred_af)
r2_af = r2_score(y_test, y_pred_af)

print("=== AUTOFEAT: LinearRegression on 20 features (10 original + 10 generated) ===\n")
print(f"  RMSE:  ${rmse_af:,.0f}")
print(f"  MAE:   ${mae_af:,.0f}")
print(f"  R²:    {r2_af:.4f}")

print(f"\n=== vs BASELINE ===")
print(f"  RMSE:  {(rmse_af - rmse_base) / rmse_base * 100:+.1f}%")
print(f"  R²:    {r2_af - r2_base:+.4f}")


=== AUTOFEAT: LinearRegression on 20 features (10 original + 10 generated) ===

  RMSE:  $39,197
  MAE:   $23,126
  R²:    0.8084

=== vs BASELINE ===
  RMSE:  -0.9%
  R²:    +0.0036


In [23]:
# Cell 15 - PyCaret Setup + Model Comparison

from pycaret.regression import setup, compare_models, tune_model, predict_model, pull

# PyCaret uses the full preprocessed df (including categoricals)
train_df = pd.concat([X_train, y_train], axis=1)

pycaret_setup = setup(
    data=train_df,
    target='SalePrice',
    session_id=42,
    verbose=False
)

print("=== PyCaret Model Comparison ===\n")
best = compare_models(include=['lr', 'rf', 'et', 'gbr', 'lightgbm'], n_select=1)
print(f"\nBest model: {type(best).__name__}")


=== PyCaret Model Comparison ===



,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
gbr,Gradient Boosting Regressor,15276.4312,583112932.3777,23623.9163,0.9032,0.1381,0.0961,0.2940
lightgbm,Light Gradient Boosting Machine,15162.3473,627854022.4152,24526.0584,0.8946,0.1357,0.0928,59.0240
et,Extra Trees Regressor,16247.9807,701273185.4529,25924.7220,0.8845,0.1487,0.1027,0.8840
rf,Random Forest Regressor,16934.9529,758777567.0747,26951.9578,0.8708,0.1507,0.1056,1.0160
lr,Linear Regression,10601105019.9042,166895118991781670158336.0000,135752412422.0945,-32565700576699.3906,0.4593,128625.5845,0.4360



Best model: GradientBoostingRegressor


In [24]:
# Cell 16 - Tune Best Model + Final Comparison

print("Tuning GradientBoostingRegressor...")
tuned = tune_model(best, verbose=False)

# Evaluate on test set
test_df = pd.concat([X_test, y_test], axis=1)
predictions = predict_model(tuned, data=test_df)

rmse_pycaret = np.sqrt(mean_squared_error(y_test, predictions['prediction_label']))
mae_pycaret = mean_absolute_error(y_test, predictions['prediction_label'])
r2_pycaret = r2_score(y_test, predictions['prediction_label'])

print("\n=== FINAL COMPARISON (Test Set) ===\n")
print(f"{'Model':45s} {'RMSE':>10s} {'MAE':>10s} {'R²':>8s}")
print("-" * 75)
print(f"{'Baseline: LinearRegression (10 features)':45s} ${rmse_base:>8,.0f} ${mae_base:>8,.0f} {r2_base:>8.4f}")
print(f"{'Autofeat: LinearRegression (20 features)':45s} ${rmse_af:>8,.0f} ${mae_af:>8,.0f} {r2_af:>8.4f}")
print(f"{'PyCaret: GBR tuned (66 features)':45s} ${rmse_pycaret:>8,.0f} ${mae_pycaret:>8,.0f} {r2_pycaret:>8.4f}")


Tuning GradientBoostingRegressor...


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Gradient Boosting Regressor,14876.6160,580220664.3808,24087.7700,0.9276,0.1135,0.0801



=== FINAL COMPARISON (Test Set) ===

Model                                               RMSE        MAE       R²
---------------------------------------------------------------------------
Baseline: LinearRegression (10 features)      $  39,568 $  24,855   0.8047
Autofeat: LinearRegression (20 features)      $  39,197 $  23,126   0.8084
PyCaret: GBR tuned (66 features)              $  24,088 $  14,877   0.9276


### Model Results

The three-way comparison tells a clear story. Autofeat's generated features barely improved the linear model (R² from 0.80 to 0.81, RMSE down by 1%). The polynomial transforms it created are redundant when the model is already linear, it can't exploit the nonlinearities they encode.

The real jump came from PyCaret's automated pipeline: R² from 0.80 to 0.93, RMSE cut by 39% ($39,568 to $24,088). Two things drove this. First, algorithm selection: Gradient Boosting naturally captures nonlinear relationships and feature interactions that LinearRegression misses. Second, the full feature set: PyCaret handled all 66 columns including categoricals (Neighborhood, Kitchen Qual, Exter Qual), which our manual baseline excluded. Several of those categorical features have strong associations with SalePrice that a numeric-only model can't access.

The lesson is that automation's value here was not in feature engineering but in model selection and preprocessing. Choosing the right algorithm mattered far more than transforming the features.


In [27]:
# Cell 17 - SHAP Feature Importance

from pycaret.regression import get_config

explainer = shap.TreeExplainer(tuned)

X_test_transformed = get_config('X_test_transformed')
shap_values = explainer.shap_values(X_test_transformed)

print("=== TOP 15 FEATURES BY MEAN |SHAP VALUE| ===\n")
feature_names = X_test_transformed.columns
mean_shap = np.abs(shap_values).mean(axis=0)
top_idx = np.argsort(mean_shap)[::-1][:15]

for i in top_idx:
    print(f"  {feature_names[i]:30s} {mean_shap[i]:>10,.0f}")


=== TOP 15 FEATURES BY MEAN |SHAP VALUE| ===

  Overall Qual                       22,680
  Gr Liv Area                        11,759
  Neighborhood                        8,944
  BsmtFin SF 1                        4,956
  Total Bsmt SF                       4,846
  Year Remod/Add                      3,520
  Garage Cars                         3,054
  Year Built                          2,966
  Overall Cond                        2,849
  1st Flr SF                          2,575
  Lot Area                            2,293
  Garage Area                         2,175
  2nd Flr SF                          1,661
  Fireplace Qu_None                   1,539
  Bsmt Full Bath                      1,443


### Key Predictors

SHAP confirms Overall Qual as the dominant predictor ($22,680 mean impact), followed by Gr Liv Area ($11,759). These two align with the Pearson correlations from EDA.

The more interesting finding is Neighborhood ranking 3rd ($8,944). This is a categorical feature that our manual baseline couldn't access, and it helps explain why PyCaret's model jumped from R²=0.80 to R²=0.93: it could leverage location information that a numeric-only model missed entirely.

Other notable features: BsmtFin SF 1 and Total Bsmt SF both rank high, confirming that basement size matters for price. Year Remod/Add outranks Year Built, suggesting buyers value recent renovations more than original construction date. And Fireplace Qu_None (the structural absence flag we created during preprocessing) carries meaningful signal, validating the decision to encode "no fireplace" explicitly rather than dropping those rows.


----------

## Reflections

### What automation improved

The most impactful automation in this workflow was PyCaret's model selection and preprocessing pipeline. It evaluated five algorithms with cross-validation, handled categorical encoding, and tuned hyperparameters, all in a few lines of code. The result was a jump from R²=0.80 (manual baseline) to R²=0.93 (tuned GBR), a 39% reduction in RMSE. Doing this manually would have meant writing separate pipelines for each algorithm, implementing encoding schemes, and running grid searches, feasible but time-consuming.

### Where automation fell short

Automated feature engineering (Autofeat) added almost nothing: R² went from 0.80 to 0.81. The polynomial transforms it generated were redundant for a linear model and unnecessary for a tree-based one that captures nonlinearities natively. Featuretools was worse, generating 992 unfiltered features that would need separate selection to be usable. Both tools ran without errors, but running a tool is not the same as adding value.

Automated EDA (YData Profiling, SweetViz) generated comprehensive reports, but the volume of output worked against us. The actionable insights came from three focused cells we wrote afterward, not from parsing hundreds of plots. The tools served best as reference material, not as the analysis itself.

### Where human judgment was essential

Every preprocessing decision required domain reasoning that no tool provided. Knowing that Pool QC's 99.6% missing values mean "no pool" rather than "data error" requires understanding the dataset, not just computing a statistic. The choice to impute Lot Frontage by Neighborhood median, to drop near-zero-variance features, to flag Garage Yr Blt=2,207 as a data entry error, these all came from interpreting the numbers, not from generating them.

Feature selection for Autofeat also required judgment: we chose the top 10 correlated features rather than feeding it all 66 columns, which would have exploded runtime without improving results.

### Honest assessment

Automation's value was concentrated in one place: model selection and tuning via PyCaret. That single step delivered nearly all of the performance improvement. Automated EDA and feature engineering were easy to run but added little that couldn't be done more effectively with targeted code. The lesson is not that automation tools are unnecessary, but that their value depends on the task. Algorithm selection and hyperparameter tuning benefit enormously from automation. EDA and feature engineering benefit more from knowing the right questions to ask.
